# Блок 11 — threading: эксперименты

Перед каждым экспериментом — **Прогноз** (до запуска), после — **Вывод** (своими словами).

In [1]:
import threading
import time

from threading_tasks import SafeCounter, fetch_all, fetch_all_manual

## Эксперимент 1. Время: 20 фейковых fetch по 0.3 с

Варианты: последовательно циклом, `fetch_all_manual`, `fetch_all(max_workers=4)`, `fetch_all(max_workers=20)`.

### Прогноз

последовательно = 20 * 0,3 – просто последовательно добавляем ссылку, поток 1; fetch_all_manual = примерно 0,3, из-за того, что 20 потоков рабоют и каждый над своей ссылкой; fetch_all(max_workers=4) – не самый быстрый, потому что слишком мало потоков, обычно их десятки и подбирают эмпирически думаю по времени примерно будет 5 * 0.3; fetch_all(max_workers=20) – примерно будет 0,3 (тоже самое, что и fetch_all_manual – но если бы было больше ссылок, то чувствую, fetch_all_manual – проиграл)


In [2]:
def fake_fetch(url: str) -> str:
    time.sleep(0.3)
    return url.upper()

urls = [f"http://site{i}.com" for i in range(20)]

variants = {
    "последовательно": lambda: [fake_fetch(u) for u in urls],
    "fetch_all_manual": lambda: fetch_all_manual(urls, fake_fetch),
    "fetch_all(max_workers=4)": lambda: fetch_all(urls, fake_fetch, max_workers=4),
    "fetch_all(max_workers=20)": lambda: fetch_all(urls, fake_fetch, max_workers=20),
}

for name, run in variants.items():
    start = time.perf_counter()
    result = run()
    elapsed = time.perf_counter() - start
    assert result == [u.upper() for u in urls]
    print(f"{name:<28} {elapsed:.2f} c")

последовательно              6.09 c
fetch_all_manual             0.31 c
fetch_all(max_workers=4)     1.53 c
fetch_all(max_workers=20)    0.31 c


### Вывод

ну в первом я был прав.


## Эксперимент 2. Гонка вживую: UnsafeCounter vs SafeCounter

8 потоков × 10 000 `increment()`, ожидаем 80 000.

### Прогноз

1. Примерно на 20-25% меньше, из-за того, что 8 потоков параллельно будут пытаться менять одно и то же значений – в итоге, потеряют в определенный момент истиное значение (потоки вытесняют друг друга)
2. 80000
3. Одинаково: замок гарантирует тот факт, что поток будет внутри блока один + еще и там, и там одинаковое число потоков


In [3]:
class UnsafeCounter:
    def __init__(self) -> None:
        self._value = 0

    def increment(self) -> None:
        tmp = self._value
        time.sleep(0)        # «отдать» GIL другому потоку между чтением и записью
        self._value = tmp + 1

    @property
    def value(self) -> int:
        return self._value


def hammer(counter, n_threads: int = 8, n_iter: int = 10_000) -> int:
    def work() -> None:
        for _ in range(n_iter):
            counter.increment()

    threads = [threading.Thread(target=work) for _ in range(n_threads)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return counter.value


expected = 8 * 10_000
for cls in (UnsafeCounter, SafeCounter):
    start = time.perf_counter()
    got = hammer(cls())
    elapsed = time.perf_counter() - start
    print(f"{cls.__name__:<14} {got:>7} / {expected}  потеряно: {expected - got:>6}  ({elapsed:.2f} c)")

UnsafeCounter    10039 / 80000  потеряно:  69961  (0.21 c)
SafeCounter      80000 / 80000  потеряно:      0  (0.01 c)


### Вывод

а во втором – по числу я был неправ, потому что интуиция подсказывает, что тут зависимость такая. У нас 80 тысяч, и 8 воркеров. Следовательно, все они пытаются работать параллельно, разделили между собой эту общую часть, и получилось ровно 10 тысяч. Ну, соответственно, из-за неправильной логики разделение это было неправильно. Именно поэтому это и получилось так. А со временем ошибся по интуиции, скорее всего, потому, что там внутри функции стоит еще sleep() Которая немного затормаживает работу.
